<a href="https://colab.research.google.com/github/NguyenDoanBinhAn1714/Lab_2-AI_IOT-bailam/blob/main/lab2ai_iot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Cài đặt các thư viện cần thiết
!pip install pandas numpy scikit-learn joblib matplotlib

import os

# Tạo cấu trúc thư mục giống hệt project mẫu
dirs = ["data", "models", "outputs/figures", "src"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Đã tạo xong cấu trúc thư mục!")

Đã tạo xong cấu trúc thư mục!


In [14]:
import pandas as pd
import numpy as np

# Tải dữ liệu công khai (dùng file training chuẩn của UCI Occupancy)
url = "https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/datatraining.txt"
try:
    df = pd.read_csv(url)
    print("Đã tải public dataset thành công.")
except:
    # Nếu không có mạng, tạo fallback sample
    print("Tải thất bại, đang tạo fallback sample...")
    dates = pd.date_range(start="2026-05-26", periods=1000, freq="min")
    df = pd.DataFrame({
        "date": dates,
        "Temperature": np.random.normal(22, 2, 1000),
        "Humidity": np.random.normal(30, 5, 1000),
        "Light": np.random.normal(400, 100, 1000),
        "CO2": np.random.normal(600, 150, 1000),
        "HumidityRatio": np.random.normal(0.004, 0.001, 1000),
        "Occupancy": np.random.choice([0, 1], 1000, p=[0.7, 0.3])
    })

# Kiểm tra schema & Làm sạch dữ liệu IoT
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Lưu telemetry_clean
df.to_csv("data/telemetry_clean.csv", index=False)

# Tạo feature dataset (ở lab này dùng trực tiếp các sensor data làm features)
features = ['Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio']
target = 'Occupancy'
df_features = df[['date'] + features + [target]].dropna()
df_features.to_csv("data/feature_dataset.csv", index=False)

print("Hoàn thành bước làm sạch và tạo feature dataset.")

Tải thất bại, đang tạo fallback sample...
Hoàn thành bước làm sạch và tạo feature dataset.


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
import json

# Chia train/test theo thời gian (không dùng random shuffle cho chuỗi thời gian)
train_size = int(len(df_features) * 0.8)
train_df = df_features.iloc[:train_size]
test_df = df_features.iloc[train_size:]

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

# Train Logistic Regression baseline
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Lưu model .joblib
joblib.dump(model, 'models/occupancy_baseline.joblib')

# Dự đoán và tính metrics
y_pred = model.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0)
}

# Lưu metrics.json
with open('outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("Đã train model, lưu model và xuất file metrics.json:")
print(metrics)

Đã train model, lưu model và xuất file metrics.json:
{'accuracy': 0.695, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


In [16]:
# Tính occupancy_probability
test_df_results = test_df.copy()
test_df_results['occupancy_probability'] = model.predict_proba(X_test)[:, 1]

# Tính anomaly_score bằng Z-score (cho tính năng CO2)
co2_mean = train_df['CO2'].mean()
co2_std = train_df['CO2'].std()
test_df_results['anomaly_score'] = (test_df_results['CO2'] - co2_mean) / co2_std

# Xác định is_anomaly (Ví dụ: Z-score > 3 hoặc < -3)
threshold_z = 3.0
test_df_results['is_anomaly'] = test_df_results['anomaly_score'].abs() > threshold_z

# Sinh decision và command_hint
def make_decision(row):
    if row['is_anomaly']:
        return "ALARM", "TURN_ON_VENTILATION"
    elif row['occupancy_probability'] > 0.5:
        return "OCCUPIED", "TURN_ON_HVAC_LIGHTS"
    else:
        return "EMPTY", "STANDBY"

test_df_results[['decision', 'command_hint']] = test_df_results.apply(
    lambda row: pd.Series(make_decision(row)), axis=1
)

# Lưu decision_log.csv
log_columns = ['date', 'occupancy_probability', 'anomaly_score', 'is_anomaly', 'decision', 'command_hint']
test_df_results[log_columns].to_csv('outputs/decision_log.csv', index=False)

print("Đã tạo decision_log.csv thành công. Dưới đây là 5 dòng mẫu:")
display(test_df_results[log_columns].head())

Đã tạo decision_log.csv thành công. Dưới đây là 5 dòng mẫu:


,date,occupancy_probability,anomaly_score,is_anomaly,decision,command_hint
800,2026-05-26 13:20:00,0.361181,0.515788,False,EMPTY,STANDBY
801,2026-05-26 13:21:00,0.359237,-1.727532,False,EMPTY,STANDBY
802,2026-05-26 13:22:00,0.313856,2.861619,False,EMPTY,STANDBY
803,2026-05-26 13:23:00,0.295513,1.485165,False,EMPTY,STANDBY
804,2026-05-26 13:24:00,0.329960,-0.032356,False,EMPTY,STANDBY


In [17]:
%%writefile src/app.py
from fastapi import FastAPI
import joblib
import pandas as pd
from pydantic import BaseModel

# Khởi tạo ứng dụng FastAPI
app = FastAPI(title="AIoT Occupancy API", description="API dự đoán trạng thái phòng cho Lab 2")

# Tải mô hình đã huấn luyện
model = joblib.load('models/occupancy_baseline.joblib')

# Định nghĩa cấu trúc dữ liệu đầu vào (Schema)
class SensorData(BaseModel):
    Temperature: float
    Humidity: float
    Light: float
    CO2: float
    HumidityRatio: float

@app.get("/")
def health_check():
    return {"status": "ok", "message": "API is up and running!"}

@app.post("/predict")
def predict_occupancy(data: SensorData):
    # Chuyển đổi dữ liệu đầu vào thành DataFrame
    df = pd.DataFrame([data.model_dump()])

    # Dự đoán xác suất
    prob = model.predict_proba(df)[0][1]

    # Đưa ra quyết định
    decision = "OCCUPIED" if prob > 0.5 else "EMPTY"

    return {
        "occupancy_probability": round(prob, 4),
        "decision": decision
    }

Overwriting src/app.py


In [18]:
%%writefile src/test_api.py
import requests

print("Đang gửi request test tới API...")

url = "http://127.0.0.1:8000/predict"
# Dữ liệu giả lập gửi lên
payload = {
    "Temperature": 24.0,
    "Humidity": 28.5,
    "Light": 500.0,
    "CO2": 850.0,
    "HumidityRatio": 0.005
}

try:
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        print("Kết quả trả về từ server:", response.json())
        print("API TEST PASSED: FastAPI model deployment is working.")
    else:
        print(f"API TEST FAILED: Status code {response.status_code}")
except Exception as e:
    print("Không kết nối được tới server. Lỗi:", e)

Overwriting src/test_api.py


In [19]:
import subprocess
import time

# 1. Cài đặt fastapi và uvicorn (nếu chưa cài)
!pip install fastapi uvicorn pydantic requests -q

# 2. Chạy Uvicorn ngầm bằng subprocess
print("Đang khởi động FastAPI server ngầm...")
server_process = subprocess.Popen(["uvicorn", "src.app:app", "--host", "127.0.0.1", "--port", "8000"])

# Đợi 3 giây để server khởi động xong
time.sleep(3)

# 3. Chạy file test_api.py để kiểm tra
print("\n--- BẮT ĐẦU TEST API ---")
!python src/test_api.py
print("------------------------\n")

# 4. Tắt server ngầm sau khi test xong để giải phóng tài nguyên
server_process.terminate()
print("Đã tắt FastAPI server.")

Đang khởi động FastAPI server ngầm...

--- BẮT ĐẦU TEST API ---
Đang gửi request test tới API...
Kết quả trả về từ server: {'occupancy_probability': 0.3309, 'decision': 'EMPTY'}
API TEST PASSED: FastAPI model deployment is working.
------------------------

Đã tắt FastAPI server.
